# Costa del Sol Flood Risk - Notebook 1: Data Processing & CRS Standardization

This notebook loads every raw dataset for the project, checks its CRS, reprojects it to the
working CRS (**EPSG:25830 - ETRS89 / UTM zone 30N**), clips it to the extent it belongs to,
and writes clean, standardized layers to `data/processed/`.

**Research question (for reference):** Which flood-prone zones of the Costa del Sol
(Tarifa-Nerja + the inland Guadalhorce valley) concentrate the greatest exposure - in built
value in urban land and in vulnerable agricultural land - and how does that risk vary with
land use and elevation above the nearest channel?

**Two clipping extents used throughout this project:**
- **Administrative extent** - the municipalities of the 3 target comarcas (Campo de
  Gibraltar `1105`, Centro-Sur/Valle del Guadalhorce `2903`, Velez-Malaga `2904`). All
  *exposure* layers (buildings, housing value, land cover, crop types, historical floods)
  are clipped here.
- **Hydrological extent** - the full river basin (`cuencas_costa_del_sol.gpkg`), which may
  extend upstream into neighbouring comarcas. All *hazard* layers (flood zones, river
  network, dams, relief) are clipped here, so distance-to-channel and elevation
  calculations aren\'t distorted near the administrative border.

This notebook does **not** compute any indicator yet (that is `2_indicators.ipynb`). It only
standardizes and stages the data. Cells marked `TODO` are for datasets not downloaded yet -
they are safe to skip and re-run later.

## 0. Setup

In [1]:
import warnings
from pathlib import Path

import geopandas as gpd
import pandas as pd

warnings.filterwarnings("ignore")

# --- Project paths (run this notebook from the project root) ---
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --- Working CRS ---
# EPSG:25830 = ETRS89 / UTM zone 30N. Projected, in metres -> correct for area
# and distance calculations. The final web map (later phase) will need
# EPSG:3857 or EPSG:4326 instead - that is a later problem, noted here on purpose.
TARGET_CRS = "EPSG:25830"

# Comarca codes that define the administrative study area
COMARCA_CODES = ["1105", "2903", "2904"]  # Campo de Gibraltar, Centro-Sur/Guadalhorce, Velez-Malaga

print(f"geopandas {gpd.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Target CRS: {TARGET_CRS}")

geopandas 1.1.3
Project root: c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk
Target CRS: EPSG:25830


## Helper functions

In [2]:
import sys


def gdal_exe(name):
    # On Windows, conda's GDAL CLI tools live in <env>/Library/bin and are
    # not always on PATH inside a Jupyter kernel - depends on how the kernel
    # got launched, even though `import rasterio`/`geopandas` works fine
    # regardless (those don't go through PATH). Resolve the exe directly
    # from the active env's prefix instead of trusting PATH; fall back to
    # the bare name for envs where it's already resolvable (Linux/macOS).
    candidate = Path(sys.prefix) / "Library" / "bin" / f"{name}.exe"
    return str(candidate) if candidate.exists() else name


def check_crs(gdf, name):
    # Print a layer\'s CRS right after reading it, before touching it.
    # Two datasets in this project (SNCZI, hi_presa_s) already turned out to be
    # in a different CRS than expected - always check first.
    print(f"[{name}] source CRS: {gdf.crs}")


def standardize(gdf, name, clip_to=None, target_crs=TARGET_CRS):
    # Reproject to target_crs (if needed) and optionally clip to a mask geometry.
    # Does not simplify geometry - that is a later, tippecanoe/web-map job.
    check_crs(gdf, name)
    if gdf.crs is None:
        raise ValueError(f"[{name}] has no CRS set - fix this before continuing")
    if gdf.crs.to_string() != target_crs:
        gdf = gdf.to_crs(target_crs)
        print(f"[{name}] reprojected -> {target_crs}")
    else:
        print(f"[{name}] already in {target_crs}, no reprojection needed")

    if clip_to is not None:
        before = len(gdf)
        gdf = gpd.clip(gdf, clip_to)
        print(f"[{name}] clipped: {before} -> {len(gdf)} features")

    return gdf


def save_layer(gdf, name):
    out_path = PROCESSED_DIR / f"{name}.gpkg"
    gdf.to_file(out_path, driver="GPKG")
    print(f"[{name}] saved -> {out_path} ({len(gdf)} features, CRS={gdf.crs})")
    return out_path


def find_file(folder, pattern):
    # Locate a file by glob pattern inside a raw-data folder, since exact
    # filenames from bulk downloads are not hardcoded here.
    matches = list(folder.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matching \'{pattern}\' in {folder}")
    if len(matches) > 1:
        print(f"WARNING: multiple matches for \'{pattern}\' in {folder}, using {matches[0]}")
    return matches[0]

## 1. Reference geometries: administrative & hydrological extents

In [3]:
admin_path = RAW_DIR / "mun_geographic_administrative_hierarchy.gpkg"
admin_all = gpd.read_file(admin_path)
check_crs(admin_all, "admin_hierarchy (raw)")
print(admin_all.columns.tolist())
admin_all.head()

[admin_hierarchy (raw)] source CRS: EPSG:4258
['Mun_Code', 'Mun_Name', 'Comarca_Code', 'Comarca_Name', 'Prov_Code', 'Prov_Name', 'CCAA_Code', 'CCAA_Name', 'nationalcode', 'geometry']


,Mun_Code,Mun_Name,Comarca_Code,Comarca_Name,Prov_Code,Prov_Name,CCAA_Code,CCAA_Name,nationalcode,geometry
0,04001,Abla,404.0,RIO NACIMIENTO,04,Almería,01,Andalucía,34010404001,"MULTIPOLYGON (((-2.78452 37.0935, -2.7841 37.0..."
1,04002,Abrucena,404.0,RIO NACIMIENTO,04,Almería,01,Andalucía,34010404002,"MULTIPOLYGON (((-2.88868 37.09169, -2.8886 37...."
2,04003,Adra,407.0,CAMPO DE DALIAS,04,Almería,01,Andalucía,34010404003,"MULTIPOLYGON (((-3.14019 36.78779, -3.13982 36..."
3,04004,Albanchez,402.0,ALTO ALMANZORA,04,Almería,01,Andalucía,34010404004,"MULTIPOLYGON (((-2.20218 37.31223, -2.20177 37..."
4,04005,Alboloduy,404.0,RIO NACIMIENTO,04,Almería,01,Andalucía,34010404005,"MULTIPOLYGON (((-2.71288 37.07817, -2.71128 37..."


**TODO before running the next cell:** confirm the comarca-code column name against the
`columns` printed above (expected: `comarca_code`). If it is called something else, update
`COMARCA_FIELD` below.

In [4]:
COMARCA_FIELD = "Comarca_Code"  # confirmed against admin_all.columns above

# Comarca_Code is stored as a float (e.g. 1105.0), so comparing it as text
# ("1105.0" vs "1105") silently matched nothing and left admin_municipios/
# admin_comarcas empty. Compare as numbers instead.
COMARCA_CODES_NUM = [int(c) for c in COMARCA_CODES]
admin_municipios = admin_all[admin_all[COMARCA_FIELD].isin(COMARCA_CODES_NUM)].copy()
admin_municipios = standardize(admin_municipios, "admin_municipios")

# Comarca-level boundary: dissolve municipios by comarca code.
# Kept as its own layer, not just a clip mask - useful for drawing comarca
# borders/zonation in the final map.
admin_comarcas = admin_municipios.dissolve(by=COMARCA_FIELD, as_index=False)
admin_comarcas = standardize(admin_comarcas, "admin_comarcas")

# Single polygon covering the whole administrative extent - used as a clip mask below
admin_extent = gpd.GeoDataFrame(geometry=[admin_municipios.union_all()], crs=admin_municipios.crs)

save_layer(admin_municipios, "admin_municipios_25830")
save_layer(admin_comarcas, "admin_comarcas_25830")


[admin_municipios] source CRS: EPSG:4258
[admin_municipios] reprojected -> EPSG:25830
[admin_comarcas] source CRS: EPSG:25830
[admin_comarcas] already in EPSG:25830, no reprojection needed
[admin_municipios_25830] saved -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\admin_municipios_25830.gpkg (62 features, CRS=EPSG:25830)
[admin_comarcas_25830] saved -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\admin_comarcas_25830.gpkg (3 features, CRS=EPSG:25830)


WindowsPath('c:/Users/juanz/OneDrive/Desktop/UCM/RURIM ESCAPE/GeoSpatial/00_Visualizaciones/costa-del-sol-flood-risk/data/processed/admin_comarcas_25830.gpkg')

In [5]:
hydro_path = RAW_DIR / "cuencas_costa_del_sol.gpkg"
hydro_cuencas = gpd.read_file(hydro_path)
hydro_cuencas = standardize(hydro_cuencas, "hydro_cuencas")

# Single polygon covering the whole basin - used as a clip mask below
hydro_extent = gpd.GeoDataFrame(geometry=[hydro_cuencas.union_all()], crs=hydro_cuencas.crs)

save_layer(hydro_cuencas, "hydro_cuencas_25830")

[hydro_cuencas] source CRS: EPSG:25830
[hydro_cuencas] already in EPSG:25830, no reprojection needed
[hydro_cuencas_25830] saved -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\hydro_cuencas_25830.gpkg (8 features, CRS=EPSG:25830)


WindowsPath('c:/Users/juanz/OneDrive/Desktop/UCM/RURIM ESCAPE/GeoSpatial/00_Visualizaciones/costa-del-sol-flood-risk/data/processed/hydro_cuencas_25830.gpkg')

## 2. Exposure layers -> clipped to the administrative extent

### 2.1 Buildings (Catastro BU) - free ATOM download, scripted below

Buildings geometry is public, no certificate needed: INSPIRE ATOM feeds per province,
one zip (GML) per municipio. The cell below downloads and unzips all 62 study municipios
directly (Cadiz=11, Malaga=29).

**Gotcha found while checking this:** 3 of our 62 municipios use a different code in
Catastro's own feed than the INE code stored in `Mun_Code` - matching by code alone
silently misses them:
- San Martin del Tesorillo: INE `11903` -> Catastro `11045`
- Malaga capital: INE `29067` -> Catastro `29900`
- Torremolinos: INE `29901` -> Catastro `29103`

(Provincial capitals and a few municipios that split off later get their own numbering in
Catastro's feed instead of the INE code.) The download cell matches by normalized name and
applies this override table, so it's handled - just flagging it here since it would be an
easy silent gap otherwise.

Building age ("antiguedad") is a separate download with its own access method - see 2.4 below.


In [6]:
import re
import zipfile

import requests

# INE code (Mun_Code) -> Catastro's own code, only where they differ (provincial
# capitals + municipios that split off later get their own numbering in the feed)
CATASTRO_CODE_OVERRIDES = {
    "11903": "11045",  # San Martin del Tesorillo
    "29067": "29900",  # Malaga capital
    "29901": "29103",  # Torremolinos
}

ATOM_URLS = {
    "11": "https://www.catastro.hacienda.gob.es/INSPIRE/buildings/11/ES.SDGC.bu.atom_11.xml",
    "29": "https://www.catastro.hacienda.gob.es/INSPIRE/buildings/29/ES.SDGC.bu.atom_29.xml",
}

ZIP_PATTERN = re.compile(
    r"https://www\.catastro\.hacienda\.gob\.es/INSPIRE/Buildings/(\d+)/(\d+)-([^/]+)/A\.ES\.SDGC\.BU\.\2\.zip"
)

buildings_folder = RAW_DIR / "buildings_catastro"
buildings_folder.mkdir(parents=True, exist_ok=True)

study = admin_municipios[["Mun_Code", "Mun_Name", "Prov_Code"]].copy()
study["catastro_code"] = study["Mun_Code"].map(lambda c: CATASTRO_CODE_OVERRIDES.get(c, c))

pending = [
    row for _, row in study.iterrows()
    if not (buildings_folder / row["Mun_Code"]).exists()
]

if not pending:
    print("Todos los municipios ya estan descargados y descomprimidos.")
else:
    for prov, atom_url in ATOM_URLS.items():
        prov_pending = [r for r in pending if r["Prov_Code"] == prov]
        if not prov_pending:
            continue
        resp = requests.get(atom_url, timeout=60)
        resp.raise_for_status()
        entry_map = {code: name for _, code, name in ZIP_PATTERN.findall(resp.text)}

        for row in prov_pending:
            code = row["catastro_code"]
            if code not in entry_map:
                print(f"NO ENCONTRADO en el feed: {row['Mun_Name']} ({row['Mun_Code']} / catastro {code})")
                continue
            feed_name = entry_map[code]
            zip_url = f"https://www.catastro.hacienda.gob.es/INSPIRE/Buildings/{prov}/{code}-{feed_name}/A.ES.SDGC.BU.{code}.zip"
            out_dir = buildings_folder / row["Mun_Code"]
            r = requests.get(zip_url, timeout=120)
            r.raise_for_status()
            zip_bytes = r.content
            out_dir.mkdir(parents=True, exist_ok=True)
            tmp_zip = buildings_folder / f"_tmp_{row['Mun_Code']}.zip"
            tmp_zip.write_bytes(zip_bytes)
            with zipfile.ZipFile(tmp_zip) as zf:
                zf.extractall(out_dir)
            tmp_zip.unlink()
            print(f"OK: {row['Mun_Name']} ({row['Mun_Code']})")

# Load every downloaded GML and concatenate into one layer
gml_files = list(buildings_folder.glob("*/*.gml"))
print(f"\n{len(gml_files)} ficheros GML encontrados en {len(list(buildings_folder.iterdir()))} carpetas de municipio")

if gml_files:
    # Tag each building with its municipio (INE Mun_Code) from the folder name
    # BEFORE concatenating - each GML lives under buildings_folder/<Mun_Code>/,
    # and that grouping is lost once everything is one big table, but
    # 2_indicators.ipynb needs Mun_Code per building (no spatial join back
    # against admin_municipios needed, and no boundary-edge ambiguity either).
    parts = []
    for f in gml_files:
        gdf = gpd.read_file(f)
        gdf["Mun_Code"] = f.parent.name
        parts.append(gdf)
    buildings = pd.concat(parts, ignore_index=True)
    buildings = gpd.GeoDataFrame(buildings, geometry="geometry")
    # No clip needed here: the GML files were downloaded one per study
    # municipio (by name, from the 62-municipio whitelist), so every building
    # already falls inside the study area - clipping against admin_extent is
    # redundant work, and gpd.clip()'s internal deep-copy is what triggered
    # the MemoryError on a 1.7M-row, 23-column table.
    buildings = standardize(buildings, "buildings")
    save_layer(buildings, "buildings_25830")
else:
    print("SKIPPED: no se encontro ningun GML - revisa la descarga arriba")


Todos los municipios ya estan descargados y descomprimidos.

186 ficheros GML encontrados en 62 carpetas de municipio
[buildings] source CRS: EPSG:25830
[buildings] already in EPSG:25830, no reprojection needed
[buildings_25830] saved -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\buildings_25830.gpkg (1714423 features, CRS=EPSG:25830)


### 2.2 Housing value (Ministerio de Vivienda) - downloaded, plain attribute table

In [7]:
# This is a plain attribute table (EUR/m2 by municipio), not a spatial layer -
# no CRS/clip step needed, just load it and inspect the sheet/header structure
# before deciding how to tidy it for the join in Notebook 2.
valor_vivienda_path = find_file(RAW_DIR, "*Valor medio de vivienda*.XLS")
print(valor_vivienda_path)

valor_vivienda_raw = pd.read_excel(valor_vivienda_path, sheet_name=None)
print("Sheets found:", list(valor_vivienda_raw.keys()))
# TODO: pick the right sheet + header row once inspected, then save a tidy
# version to data/processed/valor_vivienda.csv for Notebook 2.

c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\raw\35103500-Valor medio de vivienda libre de los municipios mayores de 25.000 habitantes.XLS
Sheets found: ['T1A2005', 'T2A2005 ', 'T3A2005 ', 'T4A2005  ', 'T1A2006 ', 'T2A2006 ', 'T3A2006 ', 'T4A2006  ', 'T1A2007 ', 'T2A2007  ', 'T3A2007 ', 'T4A2007 ', 'T1A2008 ', 'T2A2008', 'T3A2008 ', 'T4A2008', 'T1A2009', 'T2A2009', 'T3A2009', 'T4A2009', 'T1A2010', 'T2A2010 ', 'T3A2010  ', 'T4A2010', 'T1A2011', 'T2A2011', 'T3A2011', 'T4A2011', 'T1A2012', 'T2A2012', 'T3A2012', 'T4A2012', 'T1A2013', 'T2A2013', 'T3A2013', 'T4A2013', 'T1A2014', 'T2A2014', 'T3A2014', 'T4A2014', 'T1A2015', 'T2A2015', 'T3A2015', 'T4A2015', 'T1A2016', 'T2A2016', 'T3A2016', 'T4A2016', 'T1A2017', 'T2A2017', 'T3A2017', 'T4A2017', 'T1A2018', 'T2A2018', 'T3A2018', 'T4A2018', 'T1A2019', 'T2A2019', 'T3A2019', 'T4A2019', 'T1A2020', 'T2A2020', 'T3A2020', 'T4A2020', 'T1A2021', 'T2A2021', 'T3A2021', 'T4A2021', 'T1A2022', 'T2A

### 2.3 CLCplus Backbone, Imperviousness & Crop Types (Copernicus) - downloaded, raster mosaic+clip

All three come from the Copernicus Land Monitoring Service data viewer, downloaded per
province (Cadiz + Malaga) in **EPSG:3035** (LAEA Europe), 10m resolution.

- **CLCplus Backbone** (vintage 2023): tiled on the fixed LAEA 100km grid (`E28N15`,
  `E29N16`, etc), inside a `Results` subfolder per province.
- **HRL Imperviousness** (vintage 2024): same tiled LAEA grid as CLCplus, but flatter -
  `<province>/<tile>/<tile>.tif`, no `Results` subfolder.
- **Crop Types** (vintage 2023): one raster per province already (`..._ES612_0.tif` = Cadiz
  by NUTS3 code, `..._ES617_0.tif` = Malaga), no tiling.

CLCplus and Imperviousness both have the same gotcha: the Cadiz and Malaga downloads share
a few tiles at the province border (`E29N15/16/17`) - same tile, downloaded twice - so both
loaders dedupe by filename before mosaicking.

Unlike MDT02, all three are small (tens of MB total) - no reason to build a VRT here, a real
mosaicked/reprojected/clipped GeoTIFF is cheap and more convenient for the zonal-stats-style
sampling Notebook 2 will do. `-r near` on all three: these are categorical class codes
(CLCplus, Crop Types) or a bounded percentage (Imperviousness) - never interpolate class
codes; near is used for all three here for consistency, revisit for Imperviousness in
Notebook 2 if smoother values turn out to matter.


In [8]:
import subprocess

clcplus_dir = RAW_DIR / "copernicus" / "CLC Backbone 2023"
all_tiles = list(clcplus_dir.glob("*/Results/*/*.tif"))

# Cadiz and Malaga downloads overlap on the LAEA tiles at the province border -
# dedupe by filename so gdalwarp doesn't mosaic the same tile twice
unique_tiles = {t.name: t for t in all_tiles}
clcplus_tiles = sorted(unique_tiles.values())
print(f"{len(all_tiles)} tiles encontrados, {len(clcplus_tiles)} unicos tras deduplicar Cadiz/Malaga")

cutline_path = RAW_DIR / "_admin_extent_cutline.gpkg"
if not cutline_path.exists():
    admin_extent.to_file(cutline_path, driver="GPKG")

clcplus_out = PROCESSED_DIR / "clcplus_backbone_2023_25830.tif"
subprocess.run([
    gdal_exe("gdalwarp"),
    "-s_srs", "EPSG:3035",
    "-t_srs", str(TARGET_CRS),
    "-r", "near",
    "-cutline", str(cutline_path),
    "-crop_to_cutline",
    "-overwrite",
    *[str(t) for t in clcplus_tiles],
    str(clcplus_out),
], check=True)
print(f"CLCplus Backbone 2023 -> {clcplus_out}")


11 tiles encontrados, 8 unicos tras deduplicar Cadiz/Malaga
CLCplus Backbone 2023 -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\clcplus_backbone_2023_25830.tif


In [9]:
import subprocess

imperv_dir = RAW_DIR / "copernicus" / "Imperviousness 2024"
# Same tiled-LAEA-grid layout as CLCplus, but without the extra "Results"
# subfolder: <province>/<tile>/<tile>.tif
all_tiles = list(imperv_dir.glob("*/*/*.tif"))

# Same border overlap as CLCplus - Cadiz and Malaga share some tiles
unique_tiles = {t.name: t for t in all_tiles}
imperv_tiles = sorted(unique_tiles.values())
print(f"{len(all_tiles)} tiles encontrados, {len(imperv_tiles)} unicos tras deduplicar Cadiz/Malaga")

cutline_path = RAW_DIR / "_admin_extent_cutline.gpkg"
if not cutline_path.exists():
    admin_extent.to_file(cutline_path, driver="GPKG")

imperv_out = PROCESSED_DIR / "imperviousness_2024_25830.tif"
subprocess.run([
    gdal_exe("gdalwarp"),
    "-s_srs", "EPSG:3035",
    "-t_srs", str(TARGET_CRS),
    "-r", "near",
    "-cutline", str(cutline_path),
    "-crop_to_cutline",
    "-overwrite",
    *[str(t) for t in imperv_tiles],
    str(imperv_out),
], check=True)
print(f"HRL Imperviousness 2024 -> {imperv_out}")


11 tiles encontrados, 8 unicos tras deduplicar Cadiz/Malaga
HRL Imperviousness 2024 -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\imperviousness_2024_25830.tif


### 2.4 Antiguedad de edificios (Catastro CAT) - descargado con certificado/Clave PIN

Descarga masiva alfanumerica (Sede Electronica del Catastro -> "Descarga de informacion
alfanumerica", autenticado), tipologia **Urbana** solamente (Rustica tambien se descargo
pero no se usa aqui - no aporta antiguedad de edificios). Un ZIP por provincia con un
fichero `.CAT` (texto de ancho fijo, gzipeado como `.CAT.gz`) por municipio - algunos
llegaron ya descomprimidos a una carpeta con su mismo nombre.

**Formato validado contra ficheros reales, no solo contra la especificacion publica:**
- Registro tipo `11` (cabecera de finca): codigo de municipio DGC en columnas 78-80 y
  codigo de municipio **INE** en columnas 81-83 (1-based) - los dos pueden diferir.
- Registro tipo `13` (Unidad Constructiva, una fila por edificio): ano de construccion
  en columnas 296-299.

**Mismo tipo de gotcha que en 2.1 (buildings), pero con mas variantes:** el codigo DGC de
municipio no siempre coincide con el INE - capitales de provincia usan `XX900`, y algunos
municipios tienen codigo propio (San Martin del Tesorillo, Torremolinos, Malaga capital -
los mismos 3 de la nota en 2.1). Cadiz ademas archiva 8 municipios de la sierra (Jerez de
la Frontera y su entorno) bajo un prefijo de fichero totalmente distinto (`53xxx`, otra
"Gerencia" territorial historica) - ninguno de esos 8 esta en nuestros 62 municipios, pero
por eso la celda de abajo no confia en el nombre del fichero: lee la cabecera (tipo `11`)
de cada fichero para saber su codigo INE real, y solo entonces empareja con nuestros 62
municipios.

**Licencia:** solo se puede publicar el indicador agregado por municipio (media/mediana de
ano de construccion, % de edificios pre-1980), nunca el fichero CAT ni un listado por
edificio - ver nota de licencia en el plan (Fase 1).

In [10]:
import gzip
import sys

antiguedad_folder = RAW_DIR / "antiguedad_buildings_catastro"

# Adjust these two paths if you move/rename the extracted download folders -
# they're the "..._PETICION_DESCARGA_CAT/<prov>_U_.../..." folder that directly
# contains the per-municipio .CAT / .CAT.gz files (Urbana only).
CAT_URBANA_DIRS = {
    "11": antiguedad_folder / "Cadiz" / "2026090719404269534_PETICION_DESCARGA_CAT" / "11_U_23012026_CAT" / "11_U_23012026_CAT",
    "29": antiguedad_folder / "Malaga" / "2026090719444226532_PETICION_DESCARGA_CAT" / "29_U_23012026_CAT" / "29_U_23012026_CAT",
}


def winlong(path):
    # Windows' classic 260-char MAX_PATH breaks open()/CreateFileW on paths
    # this deep (OneDrive + several nested folders + the extra folder level
    # Windows Explorer's "Extract here" adds) - even though the file is
    # genuinely there and Path.iterdir()/rglob() can still see it fine. The
    # \\?\ prefix opts that one open() call into the extended-length path
    # API regardless of the registry LongPathsEnabled setting. No-op outside
    # Windows.
    if sys.platform != "win32":
        return str(path)
    resolved = str(path.resolve())
    return resolved if resolved.startswith("\\\\?\\") else "\\\\?\\" + resolved


def open_cat_entry(path):
    # A CAT entry arrives as one of three things depending on how it was
    # unzipped: a .CAT.gz file, a folder holding the extracted .CAT file
    # somewhere inside it (Windows Explorer's "Extract here" - depth can vary,
    # so search recursively rather than assuming one fixed level), or a plain
    # .CAT file.
    if path.is_dir():
        candidates = sorted(path.rglob("*.CAT"))
        if not candidates:
            raise FileNotFoundError(f"No se encontro ningun .CAT dentro de {path}")
        target = candidates[0]
    else:
        target = path
    opener = gzip.open if target.suffix == ".gz" else open
    return opener(winlong(target), "rb")


def read_cat_lines(path):
    with open_cat_entry(path) as f:
        text = f.read().decode("latin-1")
    lines = text.split("\r\n")
    if len(lines) < 5:
        lines = text.split("\n")
    return lines


# Index every CAT file by its TRUE INE municipio code, read straight from each
# file's own tipo-11 header record - not from the filename. See the markdown
# above for why the filename alone isn't reliable.
crosswalk = {}
for prov, cat_dir in CAT_URBANA_DIRS.items():
    if not cat_dir.exists():
        print(f"AVISO: no existe la carpeta esperada para provincia {prov}: {cat_dir}")
        continue
    for p in cat_dir.iterdir():
        lines = read_cat_lines(p)
        rec11 = next((l for l in lines if l[:2] == "11"), None)
        if rec11 is None:
            continue
        true_ine = prov + rec11[80:83]
        crosswalk[true_ine] = p

print(f"{len(crosswalk)} ficheros CAT indexados (Cadiz + Malaga, Urbana)")

records = []
missing = []
for _, row in admin_municipios.iterrows():
    mun_code = row["Mun_Code"]
    path = crosswalk.get(mun_code)
    if path is None:
        missing.append((mun_code, row["Mun_Name"]))
        continue
    for l in read_cat_lines(path):
        if l[:2] != "13":
            continue
        year_txt = l[295:299]
        if year_txt.isdigit():
            year = int(year_txt)
            if 1700 < year <= 2026:
                records.append((mun_code, year))

print(f"{len(records)} unidades constructivas con ano de construccion valido")
if missing:
    print(f"{len(missing)} municipios sin fichero CAT encontrado:", missing)

antiguedad_df = pd.DataFrame(records, columns=["Mun_Code", "Year_Built"])
antiguedad_by_mun = (
    antiguedad_df.groupby("Mun_Code")
    .agg(
        n_buildings_dated=("Year_Built", "count"),
        year_built_mean=("Year_Built", "mean"),
        year_built_median=("Year_Built", "median"),
        pct_pre_1980=("Year_Built", lambda s: (s < 1980).mean() * 100),
    )
    .reset_index()
)
antiguedad_by_mun.to_csv(PROCESSED_DIR / "antiguedad_by_municipio.csv", index=False)
print(antiguedad_by_mun.shape)
antiguedad_by_mun.head()

148 ficheros CAT indexados (Cadiz + Malaga, Urbana)
483939 unidades constructivas con ano de construccion valido
(62, 5)


,Mun_Code,n_buildings_dated,year_built_mean,year_built_median,pct_pre_1980
0,11004,22422,1981.579699,1982.0,45.214521
1,11008,13822,1991.767110,1993.0,13.500217
2,11013,2094,1983.426934,1985.0,44.794651
3,11021,5201,1977.231878,1975.0,52.470679
4,11022,17542,1977.556208,1979.0,50.416144


### 2.5 Crop Types - downloaded, raster mosaic+clip (see note in 2.3 above)

In [11]:
import subprocess

crop_dir = RAW_DIR / "copernicus" / "Crop Types 2023"
crop_tiles = list(crop_dir.glob("*/20230101/*.tif"))
print(f"{len(crop_tiles)} rasters de Crop Types encontrados (uno por provincia)")

cutline_path = RAW_DIR / "_admin_extent_cutline.gpkg"
if not cutline_path.exists():
    admin_extent.to_file(cutline_path, driver="GPKG")

crop_out = PROCESSED_DIR / "crop_types_2023_25830.tif"
subprocess.run([
    gdal_exe("gdalwarp"),
    "-s_srs", "EPSG:3035",
    "-t_srs", str(TARGET_CRS),
    "-r", "near",
    "-cutline", str(cutline_path),
    "-crop_to_cutline",
    "-overwrite",
    *[str(t) for t in crop_tiles],
    str(crop_out),
], check=True)
print(f"Crop Types 2023 -> {crop_out}")


2 rasters de Crop Types encontrados (uno por provincia)
Crop Types 2023 -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\crop_types_2023_25830.tif


### 2.6 Historical floods (CNIH) - Access database, read via pyodbc

`CNIH_Eventos_Inundacion_v27.accdb` is a Microsoft Access database. The `gis` conda env's
GDAL build has no ODBC/PGeo support, so `fiona`/`geopandas` cannot open it at all (both
`fiona.listlayers()` and opening a specific layer directly fail with "unsupported driver:
ODBC"). QGIS's own GDAL build can list the tables, but none of them carry usable geometry
either.

Turns out geometry isn't needed: `CNIH_EVENTO` itself has no coordinates (only a
whole river-basin-district code), but `CNIH_DAÑO_ECONOMICO_MUNICIPIO` and
`CNIH_DAÑO_PERSONA_MUNICIPIO` link each event straight to a municipio via a 5-digit INE
code (`CDEM_COD_MUNICIPIO` / `CDPM_COD_MUNICIPIO` - same format as `Mun_Code` above). So
this is read with `pyodbc` (Windows' native Access driver, works fine outside GDAL) and
joined to the study municipios by attribute - no spatial join at all.


In [12]:
import pyodbc

cnih_path = RAW_DIR / "CNIH_Eventos_Inundacion_v27" / "CNIH_Eventos_Inundacion_v27.accdb"

conn_str = (
    r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
    rf"DBQ={cnih_path};"
)
conn = pyodbc.connect(conn_str)

econ = pd.read_sql(
    "SELECT CE_COD_EVENTO, CDEM_COD_MUNICIPIO, CDEM_NOM_MUNICIPIO, "
    "CDEM_NIVEL_DAÑO, CDEM_INDEMNIZACION_CCS FROM CNIH_DAÑO_ECONOMICO_MUNICIPIO",
    conn,
)
pers = pd.read_sql(
    "SELECT CE_COD_EVENTO, CDPM_COD_MUNICIPIO, CDPM_NOM_MUNICIPIO, "
    "CDPM_NIVEL_DAÑO, CDPM_VICTIMAS, CDPM_EVACUADOS, CDPM_RESCATADOS "
    "FROM CNIH_DAÑO_PERSONA_MUNICIPIO",
    conn,
)
conn.close()

# Same Mun_Code field as admin_municipios above - both are 5-digit INE codes
study_codes = set(admin_municipios["Mun_Code"].astype(str).str.zfill(5))

econ["CDEM_COD_MUNICIPIO"] = econ["CDEM_COD_MUNICIPIO"].astype(str).str.zfill(5)
pers["CDPM_COD_MUNICIPIO"] = pers["CDPM_COD_MUNICIPIO"].astype(str).str.zfill(5)

# Union event-municipio pairs from both tables - an event can be logged in
# one table (e.g. only economic damage) but not the other, so counting from
# just one table would undercount events per municipio.
econ_pairs = econ.rename(
    columns={"CDEM_COD_MUNICIPIO": "cod_municipio", "CDEM_NOM_MUNICIPIO": "nom_municipio"}
)[["CE_COD_EVENTO", "cod_municipio", "nom_municipio"]]
pers_pairs = pers.rename(
    columns={"CDPM_COD_MUNICIPIO": "cod_municipio", "CDPM_NOM_MUNICIPIO": "nom_municipio"}
)[["CE_COD_EVENTO", "cod_municipio", "nom_municipio"]]
event_mun = pd.concat([econ_pairs, pers_pairs]).drop_duplicates(subset=["CE_COD_EVENTO", "cod_municipio"])

# CNIH covers the whole of Spain - filter down to the study municipios
event_mun = event_mun[event_mun["cod_municipio"].isin(study_codes)]

events_per_mun = (
    event_mun.groupby(["cod_municipio", "nom_municipio"])
    .size()
    .reset_index(name="n_eventos")
)

# Economic severity for the same municipios, as extra signal (not required,
# but cheap to carry forward for Notebook 2 to use or ignore)
econ_study = econ[econ["CDEM_COD_MUNICIPIO"].isin(study_codes)]
econ_agg = (
    econ_study.groupby("CDEM_COD_MUNICIPIO")
    .agg(
        indemnizacion_total_eur=("CDEM_INDEMNIZACION_CCS", "sum"),
        nivel_dano_max=("CDEM_NIVEL_DAÑO", "max"),
    )
    .reset_index()
    .rename(columns={"CDEM_COD_MUNICIPIO": "cod_municipio"})
)

cnih_by_municipio = events_per_mun.merge(econ_agg, on="cod_municipio", how="left")

out_path = PROCESSED_DIR / "cnih_events_by_municipio.csv"
cnih_by_municipio.to_csv(out_path, index=False)
print(f"Municipios con eventos CNIH en la zona de estudio: {len(cnih_by_municipio)}")
cnih_by_municipio.sort_values("n_eventos", ascending=False).head(10)


Municipios con eventos CNIH en la zona de estudio: 24


,cod_municipio,nom_municipio,n_eventos,indemnizacion_total_eur,nivel_dano_max
13,29067,Málaga,16,1805168.92,NaN
14,29069,Marbella,5,2190056.67,NaN
15,29070,Mijas,5,2563812.88,NaN
1,11008,Los Barrios,4,NaN,NaN
9,29042,Coín,2,39622.45,NaN
0,11004,Algeciras,2,18719.43,NaN
19,29082,Rincón de la Victoria,2,9092561.44,NaN
10,29051,Estepona,2,25617.17,NaN
21,29094,Vélez-Málaga,2,170968.04,NaN
6,29008,Alhaurín el Grande,1,107512.25,NaN


## 3. Hazard layers -> clipped to the hydrological extent

### 3.1 SNCZI flood zones (T10 / T50 / T100 / T500)

In [13]:
snczi_folders = {
    "T10": "laminasPB-q10",
    "T50": "laminas-q50",
    "T100": "laminasPB-q100",
    "T500": "laminasPB-q500",
}

flood_zones = {}
for period, folder_name in snczi_folders.items():
    folder = RAW_DIR / folder_name
    shp_path = find_file(folder, "*.shp")
    gdf = gpd.read_file(shp_path)
    gdf = standardize(gdf, f"snczi_{period}", clip_to=hydro_extent)
    save_layer(gdf, f"snczi_{period.lower()}_25830")
    flood_zones[period] = gdf

# Note: laminasPB-q100/ has a stray .sbn file with a "PBC" suffix (no matching
# .shp) - the glob above only picks up the real .shp, so this is safe to ignore.

[snczi_T10] source CRS: EPSG:25830
[snczi_T10] already in EPSG:25830, no reprojection needed
[snczi_T10] clipped: 4966 -> 110 features
[snczi_t10_25830] saved -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\snczi_t10_25830.gpkg (110 features, CRS=EPSG:25830)
[snczi_T50] source CRS: EPSG:25830
[snczi_T50] already in EPSG:25830, no reprojection needed
[snczi_T50] clipped: 2578 -> 111 features
[snczi_t50_25830] saved -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\snczi_t50_25830.gpkg (111 features, CRS=EPSG:25830)
[snczi_T100] source CRS: EPSG:25830
[snczi_T100] already in EPSG:25830, no reprojection needed
[snczi_T100] clipped: 5115 -> 111 features
[snczi_t100_25830] saved -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\snczi_t100_25830.gpkg (111 features, CRS=EPSG:258

### 3.2 Curated hydrography (already merged/curated in QGIS)

In [14]:
# Curation already closed (see plan doc): these 4 layers are the final set -
# hi_tramocurso_l + hi_presa_l feed the analysis directly, hi_tramocurso_s +
# hi_zhumeda_s are map/context only.
hidro_layers = {
    "hi_tramocurso_l": "hi_tramocurso_l_costadelsol.gpkg",
    "hi_presa_l": "hi_presa_l_costadelsol.gpkg",
    "hi_tramocurso_s": "hi_tramocurso_s_costadelsol.gpkg",
    "hi_zhumeda_s": "hi_zhumeda_s_costadelsol.gpkg",
}

for name, filename in hidro_layers.items():
    gdf = gpd.read_file(RAW_DIR / filename)
    gdf = standardize(gdf, name, clip_to=hydro_extent)
    save_layer(gdf, f"{name}_25830")

[hi_tramocurso_l] source CRS: EPSG:4258
[hi_tramocurso_l] reprojected -> EPSG:25830
[hi_tramocurso_l] clipped: 19928 -> 19928 features
[hi_tramocurso_l_25830] saved -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\hi_tramocurso_l_25830.gpkg (19928 features, CRS=EPSG:25830)
[hi_presa_l] source CRS: EPSG:4258
[hi_presa_l] reprojected -> EPSG:25830
[hi_presa_l] clipped: 88 -> 88 features
[hi_presa_l_25830] saved -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\hi_presa_l_25830.gpkg (88 features, CRS=EPSG:25830)
[hi_tramocurso_s] source CRS: EPSG:4258
[hi_tramocurso_s] reprojected -> EPSG:25830
[hi_tramocurso_s] clipped: 151 -> 151 features
[hi_tramocurso_s_25830] saved -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\hi_tramocurso_s_25830.gpkg (151 features, CRS=EPSG:25830)

### 3.3 Relief - MDT02

Two things to handle before this can be sampled correctly: the tiles are split across UTM
zone 29N (`HU29`) and 30N (`HU30`) because the study area straddles the 6-deg-W meridian, and
there is a full duplicate set of every tile in WGS84 (only the `ETRS89` ones are used below).

This does **not** build a materialized mosaic/clip - see the note in the VRT cell below for
why. The zone-29 tiles still need to be reprojected individually first (each tile is small,
that part is cheap); it is only merging everything into one big raster that is not worth doing.

In [15]:
import rasterio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask as rio_mask

MDT_TARGET_CRS = "EPSG:25830"
mdt_folder = RAW_DIR / "MDT02"

# Only the ETRS89 tiles - the "-WGS84-" files look like a duplicate export of
# the same tiles in a different datum (TODO: confirm with Juan whether those
# can just be deleted).
etrs89_tiles = sorted(mdt_folder.glob("MDT02-ETRS89-*.tif"))
zone29_tiles = [t for t in etrs89_tiles if "-HU29-" in t.name]
zone30_tiles = [t for t in etrs89_tiles if "-HU30-" in t.name]

print(f"Found {len(etrs89_tiles)} ETRS89 tiles total")
print(f"Zone 29N tiles (need reprojection to 25830): {len(zone29_tiles)}")
print(f"Zone 30N tiles (already in 25830):           {len(zone30_tiles)}")

Found 100 ETRS89 tiles total
Zone 29N tiles (need reprojection to 25830): 2
Zone 30N tiles (already in 25830):           98


In [16]:
def reproject_tile(src_path, dst_path, dst_crs=MDT_TARGET_CRS):
    with rasterio.open(src_path) as src:
        if src.crs.to_string() == dst_crs:
            return
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({"crs": dst_crs, "transform": transform, "width": width, "height": height})
        with rasterio.open(dst_path, "w", **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.bilinear,
                )

REPROJECTED_DIR = RAW_DIR / "MDT02_reprojected_25830"
REPROJECTED_DIR.mkdir(exist_ok=True)

mosaic_inputs = list(zone30_tiles)  # already in the right CRS
for tile in zone29_tiles:
    out_path = REPROJECTED_DIR / tile.name
    if not out_path.exists():
        reproject_tile(tile, out_path)
    mosaic_inputs.append(out_path)

print(f"{len(mosaic_inputs)} tiles ready for mosaic, all in {MDT_TARGET_CRS}")

100 tiles ready for mosaic, all in EPSG:25830


In [17]:
# Build a VRT (virtual raster) instead of materializing one big mosaic file.
# A VRT is just a small XML pointer to the underlying tiles - instant to
# build, and rasterio can sample elevation values from it exactly like a
# real raster, reading only the tiles/blocks a given query actually touches.
#
# Why not a materialized, clipped GeoTIFF like the other layers: the basins
# in cuencas_costa_del_sol.gpkg are scattered end-to-end along ~150km of
# coast (Tarifa to Nerja), so their combined bounding box is nearly as big
# as the whole project bbox, even though the actual basin area is much
# smaller. A full-resolution (2m) raster covering that bounding rectangle
# is tens of GB - not worth materializing when what Notebook 2 actually
# needs is point elevation values (building/cell centroids), not a picture.
# If a low-res relief basemap for the final map turns out to be wanted,
# that gets built separately, at a coarse resolution, in 3_web_export.ipynb.
import subprocess

vrt_path = PROCESSED_DIR / "relief_mdt02_25830.vrt"
tile_list_path = RAW_DIR / "_mdt_tile_list.txt"
tile_list_path.write_text(chr(10).join(str(p) for p in mosaic_inputs))

subprocess.run([
    gdal_exe("gdalbuildvrt"),
    "-input_file_list", str(tile_list_path),
    str(vrt_path),
], check=True)

print(f"VRT built -> {vrt_path}")
print("Notebook 2 samples elevation directly from this VRT (rasterio .sample()) -")
print("no separate mosaic/clip step needed.")

VRT built -> c:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\costa-del-sol-flood-risk\data\processed\relief_mdt02_25830.vrt
Notebook 2 samples elevation directly from this VRT (rasterio .sample()) -
no separate mosaic/clip step needed.


## 4. CRS verification summary

In [18]:
mdt_vrt_path = PROCESSED_DIR / "relief_mdt02_25830.vrt"
if mdt_vrt_path.exists():
    with rasterio.open(mdt_vrt_path) as src:
        status = "OK" if src.crs.to_string() == TARGET_CRS else "MISMATCH"
        print(f"{mdt_vrt_path.stem:40s} {str(src.crs):15s} {'vrt':>8s}  [{status}]")

relief_mdt02_25830                       EPSG:25830           vrt  [OK]


## Summary & handoff to Notebook 2

Everything successfully processed above now lives in `data/processed/` as GeoPackages, all in
**EPSG:25830**, clipped to the extent that applies to it (administrative for exposure,
hydrological for hazard) - except relief, which is a VRT (virtual raster, sampled on demand in
Notebook 2) rather than a materialized/clipped file.

Still pending (see `capstone_costa_del_sol.md` for status/links): buildings and building age
(Catastro), the three Copernicus layers, and picking the right CNIH table. Re-run the
relevant section above once each is sorted out.

Kept on purpose for `2_indicators.ipynb`:
- Municipio/comarca code on every exposure layer, for the choropleth join.
- Crop type code on Crop Types (once downloaded), for the agricultural-lane breakdown.
- Geometries left at full detail - no simplification here, that happens later (tippecanoe /
  web map step).

**CRS reminder for later:** everything here is EPSG:25830 for correct area/distance math.
The final web map will need EPSG:3857 or EPSG:4326 - that conversion happens in the web map
step, not here.